# Car Price Prediction — Exploratory Data Analysis

This notebook performs a complete EDA on the CarDekho dataset before any
feature engineering or modeling happens. The goal of EDA is to understand
the data's quality, distributions, and relationships **before** trusting
it to a model — skipping this step is the #1 reason production models
underperform silently.

This notebook is written to be **schema-adaptive**: it inspects whatever
numeric/categorical columns actually exist in the CSV, rather than
hardcoding exact column names. That way it works whether you're using
the classic 8-column CarDekho dataset or the newer extended version.

Sections:
1. Load data
2. Missing values
3. Duplicate values
4. Data types
5. Distribution plots
6. Correlation analysis
7. Pair plots
8. Outlier detection
9. Target variable analysis


## 1. Setup & Load Data

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_ingestion import DataIngestion

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

ingestion = DataIngestion()
df = ingestion.load_data()

print(f"Shape: {df.shape}")
df.head()


## 2. Missing Values

We check both the **count** and **percentage** of missing values per
column. A column with, say, 40%+ missing values needs a different
strategy (drop it, or a flag + imputation) than one with 1% missing
(simple imputation is usually safe).

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
}).sort_values("missing_count", ascending=False)

missing_summary


In [ ]:
# Visualize missingness — a fully white column means "no missing data"
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing Value Heatmap")
plt.show()


## 3. Duplicate Values

Duplicate rows inflate certain patterns and can leak identical records
across train/test splits, artificially boosting validation scores.

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

if num_duplicates > 0:
    display(df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist()))


## 4. Data Types

Confirms which columns are numeric vs categorical (object/string) so
the rest of the notebook — and later, `feature_engineering.py` — treats
each column correctly.

In [ ]:
df.info()


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


## 5. Distribution Plots

Histograms for every numeric column reveal skewness, scale differences,
and potential outliers at a glance — all of which inform whether we need
log-transforms or scaling later.

In [ ]:
n_cols = len(numeric_cols)
fig, axes = plt.subplots(nrows=(n_cols + 2) // 3, ncols=3, figsize=(15, 4 * ((n_cols + 2) // 3)))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribution of {col}")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
# Categorical value counts
for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=len(categorical_cols), figsize=(6 * len(categorical_cols), 5))
if len(categorical_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, categorical_cols):
    sns.countplot(data=df, x=col, ax=ax, palette="viridis")
    ax.set_title(f"Count of {col}")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## 6. Correlation Analysis

Correlation between numeric features and the target tells us which
features are likely to matter most, and correlation *between* features
flags multicollinearity — important for linear models like Ridge/Lasso
which are sensitive to it (tree-based models are largely immune).

In [ ]:
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()


## 7. Pair Plots

Pair plots visualize pairwise relationships between all numeric
variables simultaneously — useful for spotting non-linear relationships
that a correlation coefficient (which only measures *linear* association)
would miss entirely.

In [ ]:
sns.pairplot(df[numeric_cols], diag_kind="kde", corner=True)
plt.suptitle("Pair Plot of Numeric Features", y=1.02)
plt.show()


## 8. Outlier Detection

We use the IQR (Interquartile Range) method: any point beyond
`Q1 - 1.5*IQR` or `Q3 + 1.5*IQR` is flagged as an outlier. Boxplots
visualize this directly. We only **flag** outliers here — the decision
on whether to cap, remove, or keep them happens in
`feature_engineering.py`, with reasoning documented there.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=len(numeric_cols), figsize=(4 * len(numeric_cols), 5))
if len(numeric_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=df, y=col, ax=ax, color="tomato")
    ax.set_title(col)

plt.tight_layout()
plt.show()


In [ ]:
def iqr_outlier_summary(df, cols):
    rows = []
    for col in cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        rows.append({
            "column": col,
            "lower_bound": round(lower, 2),
            "upper_bound": round(upper, 2),
            "num_outliers": n_outliers,
            "pct_outliers": round(n_outliers / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values("num_outliers", ascending=False)

iqr_outlier_summary(df, numeric_cols)


## 9. Target Variable Analysis

Understanding the target (`Selling_Price`) distribution is critical:
if it's heavily right-skewed (common for price data), a log-transform
during training can significantly improve linear model performance and
stabilize variance.

In [ ]:
target_col = [c for c in df.columns if c.strip().lower() == "selling_price"][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[target_col], kde=True, ax=axes[0], color="seagreen")
axes[0].set_title(f"Distribution of {target_col}")

sns.boxplot(y=df[target_col], ax=axes[1], color="seagreen")
axes[1].set_title(f"Boxplot of {target_col}")

plt.tight_layout()
plt.show()

print(f"Skewness: {df[target_col].skew():.3f}")
print(f"Kurtosis: {df[target_col].kurt():.3f}")


In [ ]:
# Relationship between target and each categorical feature
for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=col, y=target_col, palette="viridis")
    plt.title(f"{target_col} by {col}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## Summary of EDA Findings

*(Fill this in once run against the real dataset — this is the section
you'd present in a portfolio/interview to show you can draw conclusions,
not just generate plots.)*

- Missing values: ...
- Duplicates: ...
- Skewed features needing transformation: ...
- Strongest predictors of `Selling_Price`: ...
- Outlier handling decision: ...
